# Assignment 4: DML with Random Forest (outcome) + ElasticNet (treatment)
LinearDML from EconML, 5-fold cross-fitting. Controls (W) are the DAG4 adjustment sets from Assignment 3.
X is the 15 predictors that are not one of the five causes (same setup as the group's shared code).
Put `StressLevelDataset.csv` in the same folder as this notebook.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from econml.dml import LinearDML

## Setup: data, outcome, causes, adjustment sets

In [2]:
data = pd.read_csv('StressLevelDataset.csv')
OUTCOME = 'stress_level'
CAUSES = ['study_load','noise_level','living_conditions','academic_performance','future_career_concerns']

# Adjustment sets from DAG4 (Assignment 3)
ADJ = {'study_load': [], 'living_conditions': [],
       'noise_level': ['living_conditions'],
       'academic_performance': ['living_conditions','study_load'],
       'future_career_concerns': ['academic_performance']}

# X = every predictor that is not one of the 5 causes (15 columns)
X_COLS = [c for c in data.columns if c != OUTCOME and c not in CAUSES]
print(len(X_COLS), 'X columns:', X_COLS)

15 X columns: ['anxiety_level', 'self_esteem', 'mental_health_history', 'depression', 'headache', 'blood_pressure', 'sleep_quality', 'breathing_problem', 'safety', 'basic_needs', 'teacher_student_relationship', 'social_support', 'peer_pressure', 'extracurricular_activities', 'bullying']


## My two models

In [3]:
def make_rf():      # outcome model
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                 n_jobs=-1, random_state=0)

def make_enet():    # treatment model
    return make_pipeline(StandardScaler(),
                         ElasticNetCV(l1_ratio=[.1,.5,.9,1], cv=5, random_state=0, max_iter=5000))

## Group evaluation code (unchanged)
DML fit, swap test, placebo test, and cross-validated R² of both nuisance models.

In [4]:
def _fit_dml(make_y, make_t, T, W_cols):
    X = data[X_COLS].values
    est = LinearDML(model_y=make_y(), model_t=make_t(),
                    discrete_treatment=False, cv=5, random_state=0)
    est.fit(data[OUTCOME].values, T, X=X,
            W=data[W_cols].values if W_cols else None)
    ate = float(np.ravel(est.ate(X))[0])
    lo, hi = est.ate_interval(X, alpha=0.05)
    return ate, float(np.ravel(lo)[0]), float(np.ravel(hi)[0])

def evaluate_model(name, make_y, make_t, reps=3, seed=0):
    rng = np.random.default_rng(seed); rows = []
    for cause in CAUSES:
        W_cols, T = ADJ[cause], data[cause].values
        ate, lo, hi = _fit_dml(make_y, make_t, T, W_cols)
        ate_swap, _, _ = _fit_dml(make_t, make_y, T, W_cols)                       # roles swapped
        plc = [_fit_dml(make_y, make_t, rng.permutation(T), W_cols) for _ in range(reps)]  # placebo
        cols = W_cols + X_COLS
        r2_y = cross_val_score(make_y(), data[cols], data[OUTCOME], cv=5, scoring='r2').mean()
        r2_t = cross_val_score(make_t(), data[cols], data[cause], cv=5, scoring='r2').mean()
        rows.append({'model': name, 'cause': cause, 'ATE': ate, 'CI_low': lo, 'CI_high': hi,
                     'swap_diff': abs(ate-ate_swap),
                     'placebo_abs': np.mean([abs(p[0]) for p in plc]),
                     'placebo_cover': np.mean([p[1] <= 0 <= p[2] for p in plc]),
                     'r2_y': r2_y, 'r2_t': r2_t})
        print('done:', cause, flush=True)
    return pd.DataFrame(rows)

## Run it

In [5]:
results = evaluate_model('RF+ENet', make_rf, make_enet)
results.to_csv('results_sabayna.csv', index=False)
results.round(4)

done: study_load


done: noise_level


done: living_conditions


done: academic_performance


done: future_career_concerns


,model,cause,ATE,CI_low,CI_high,swap_diff,placebo_abs,placebo_cover,r2_y,r2_t
0,RF+ENet,study_load,0.0137,-0.0085,0.0360,0.0003,0.0056,1.0000,0.7940,0.4482
1,RF+ENet,noise_level,0.0199,-0.0009,0.0406,0.0039,0.0118,0.6667,0.7891,0.4692
2,RF+ENet,living_conditions,0.0252,-0.0056,0.0560,0.0026,0.0030,1.0000,0.7940,0.4185
3,RF+ENet,academic_performance,-0.0308,-0.0636,0.0020,0.0036,0.0063,1.0000,0.7933,0.6024
4,RF+ENet,future_career_concerns,-0.0399,-0.1158,0.0359,0.0034,0.0056,1.0000,0.7921,0.6842


## Reading the results
- No effect is significant: every 95% interval includes zero.
- Placebo coverage is 1.00 for four causes. `noise_level` is 0.67, so that one is the least reliable.
- Swap differences are tiny, so estimates are stable when the outcome and treatment models are exchanged.
- The outcome model fits well (r2_y around 0.79) and the treatment model reasonably (r2_t 0.42-0.68).